# Wildfire Analysis — Master Notebook

## [ENG] Overview

This is the main notebook for the Wildfire Analysis & ML Pipeline project.
It provides a complete, end-to-end walkthrough of the entire workflow:

1. **Data ingestion** — loading FIRMS fire detections, CLCPlus land cover, Open-Meteo weather
2. **Sensor merging** — combining VIIRS and MODIS into a unified dataset
3. **Confidence mapping** — reconciling VIIRS numerical and MODIS categorical confidence
4. **CLC enrichment** — adding land cover classification to each fire detection
5. **Weather enrichment** — adding historical weather data to each fire detection
6. **Analysis & visualization** — exploring the enriched dataset
7. **Modeling** — training and evaluating the ML model

## [ESP] Descripción general

Este es el notebook principal del proyecto Wildfire Analysis & ML Pipeline.
Proporciona un recorrido completo, de principio a fin, de todo el flujo de trabajo:

1. **Ingesta de datos** — carga de detecciones de fuego FIRMS, cobertura del suelo CLCPlus, clima Open-Meteo
2. **Fusión de sensores** — combinación de VIIRS y MODIS en un dataset unificado
3. **Mapeo de confianza** — reconciliación de confianza numérica VIIRS y categórica MODIS
4. **Enriquecimiento CLC** — adición de clasificación de cobertura del suelo a cada detección
5. **Enriquecimiento climático** — adición de datos de clima histórico a cada detección
6. **Análisis y visualización** — exploración del dataset enriquecido
7. **Modelado** — entrenamiento y evaluación del modelo ML

## [ENG] Project structure / [ESP] Estructura del proyecto

```text
data/raw/firms/          → Raw FIRMS CSVs (Spain/2023, Spain/2024)
data/raw/clcplus/        → CLCPlus GeoTIFF tiles (Spain/2023-2025)
data/processed/merged/   → VIIRS+MODIS merged output
data/processed/enriched/ → Enriched datasets (CLC + weather)
data/processed/predictions/ → Model predictions
src/wildfire/            → Python package (data loading, enrichment, processing)
configs/                 → Configuration files
```

## [ENG] Supporting notebooks / [ESP] Notebooks de soporte

| Notebook | [ENG] Purpose | [ESP] Propósito |
|---|---|---|
| `01_data_exploration.ipynb` | Explore raw FIRMS, CLCPlus, and weather data | Explorar datos crudos FIRMS, CLCPlus y clima |
| `02_enrichment.ipynb` | Walk through the enrichment process step by step | Recorrer el proceso de enriquecimiento paso a paso |
| `03_analysis.ipynb` | Statistical analysis and feature engineering | Análisis estadístico e ingeniería de características |
| `04_modeling.ipynb` | ML model training, evaluation, and predictions | Entrenamiento, evaluación y predicciones del modelo ML |

In [25]:
import pandas as pd

"""NASA FIRMS data loading utilities."""

from __future__ import annotations

from pathlib import Path

import pandas as pd

from wildfire.config import load_config

VIIRS_COLUMNS = {
    "latitude": float,
    "longitude": float,
    "bright_ti4": float,
    "scan": float,
    "track": float,
    "acq_date": str,
    "acq_time": str,
    "satellite": str,
    "confidence": str,
    "version": str,
    "bright_ti5": float,
    "frp": float,
    "daynight": str,
}

MODIS_COLUMNS = {
    "latitude": float,
    "longitude": float,
    "brightness": float,
    "scan": float,
    "track": float,
    "acq_date": str,
    "acq_time": str,
    "satellite": str,
    "confidence": str,
    "version": str,
    "bright_t31": float,
    "frp": float,
    "daynight": str,
}

VIIRS_COLS_TO_RENAME = {
    "bright_ti4": "brightness",
    "bright_ti5": "brightness_ir",
}

MODIS_COLS_TO_RENAME = {
    "bright_t31": "brightness_ir",
}

SENSOR_FOLDER_NAMES = {
    "modis": "MODIS",
    "viirs_noaa20": "VIIRS/VIIRS NOAA-20",
    "viirs_snpp": "VIIRS/VIIRS S-NPP",
}


def _firms_dir(country: str, year: int, sensor: str) -> Path:
    """Build the directory path for a FIRMS sensor/year/country."""
    from wildfire.config import PROJECT_ROOT
    config = load_config()
    sensor_path = SENSOR_FOLDER_NAMES.get(sensor.lower(), sensor)
    return PROJECT_ROOT / config["data"]["raw"] / "firms" / country / str(year) / sensor_path


def _find_csv_in_dir(directory: Path) -> Path | None:
    """Find the first CSV file in a directory."""
    if not directory.exists():
        return None
    csvs = list(directory.glob("*.csv"))
    return csvs[0] if csvs else None


def load_firms(
    country: str = "Spain",
    year: int = 2023,
    sensor: str = "viirs_snpp",
) -> pd.DataFrame:
    """Load a single FIRMS CSV file and normalise column names.

    Parameters
    ----------
    country:
        Country name (must match folder name under ``data/raw/firms/``).
    year:
        Four-digit year.
    sensor:
        One of ``"modis"``, ``"viirs_snpp"``, ``"viirs_noaa20"``.

    Returns
    -------
    pd.DataFrame
        Normalised DataFrame with a ``sensor`` column added.

    Raises
    ------
    ValueError
        If ``sensor`` is not recognised.
    FileNotFoundError
        If the CSV file does not exist.
    """
    sensor = sensor.lower()

    if sensor not in SENSOR_FOLDER_NAMES:
        valid = ", ".join(sorted(SENSOR_FOLDER_NAMES.keys()))
        raise ValueError(f"Unknown sensor: {sensor!r}. Use one of: {valid}")

    directory = _firms_dir(country, year, sensor)
    csv_path = _find_csv_in_dir(directory)

    if csv_path is None:
        raise FileNotFoundError(f"No FIRMS CSV found in {directory}")

    df = pd.read_csv(csv_path)

    df["sensor"] = sensor
    df["country"] = country
    df["year"] = year

    if sensor.startswith("viirs"):
        df.rename(columns=VIIRS_COLS_TO_RENAME, inplace=True)
    elif sensor == "modis":
        df.rename(columns=MODIS_COLS_TO_RENAME, inplace=True)

    return df


def load_all_firms(
    country: str = "Spain",
    years: list[int] | None = None,
    sensors: list[str] | None = None,
) -> pd.DataFrame:
    """Load FIRMS data for all specified countries, years, and sensors.

    Parameters
    ----------
    country:
        Country name.
    years:
        List of years. Defaults to configured years.
    sensors:
        List of sensors. Defaults to ``["modis", "viirs_snpp", "viirs_noaa20"]``.

    Returns
    -------
    pd.DataFrame
        Concatenated FIRMS data from all specified combinations.
    """
    if years is None:
        config = load_config()
        years = config["firms"]["years"]
    if sensors is None:
        sensors = ["modis", "viirs_snpp", "viirs_noaa20"]

    frames: list[pd.DataFrame] = []
    for year in years:
        for sensor in sensors:
            try:
                df = load_firms(country=country, year=year, sensor=sensor)
                frames.append(df)
            except FileNotFoundError:
                continue

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


def list_available_firms(country: str = "Spain") -> list[dict[str, str | int]]:
    """List which FIRMS files exist on disk.

    Walks the folder hierarchy under ``data/raw/firms/{country}/{year}/`` and
    derives the sensor key from the folder name rather than parsing filenames.

    Returns
    -------
    list[dict]
        Each dict has keys ``country``, ``year``, ``sensor``, ``path``.
    """
    config = load_config()
    base_dir = Path(config["data"]["raw"]) / "firms" / country
    results: list[dict[str, str | int]] = []

    if not base_dir.exists():
        return results

    folder_to_sensor = {v.lower(): k for k, v in SENSOR_FOLDER_NAMES.items()}

    for year_dir in sorted(base_dir.iterdir()):
        if not year_dir.is_dir():
            continue
        try:
            year = int(year_dir.name)
        except ValueError:
            continue
        for csv_path in sorted(year_dir.rglob("*.csv")):
            # csv_path is like .../{year}/MODIS/file.csv or .../{year}/VIIRS/VIIRS S-NPP/file.csv
            # Use relative path from year_dir to match SENSOR_FOLDER_NAMES keys
            rel_dir = csv_path.parent.relative_to(year_dir).as_posix().lower()
            sensor_key = folder_to_sensor.get(rel_dir, rel_dir)
            results.append({
                "country": country,
                "year": year,
                "sensor": sensor_key,
                "path": str(csv_path),
            })

    return results


"""Confidence mapping between MODIS numerical and VIIRS categorical values."""

from __future__ import annotations

from pathlib import Path

import pandas as pd
import yaml

from wildfire.config import load_config
from wildfire.data.firms import load_all_firms


def _load_thresholds() -> dict:
    """Load the confidence thresholds YAML file."""
    from wildfire.config import PROJECT_ROOT
    config = load_config()
    path = PROJECT_ROOT / config["paths"]["confidence_thresholds"]
    with open(path) as f:
        return yaml.safe_load(f)


def numerical_to_categorical(value: float, thresholds: dict | None = None) -> str:
    """Convert a MODIS numerical confidence (0-100) to a categorical label.

    Parameters
    ----------
    value:
        Numerical confidence value.
    thresholds:
        Optional pre-loaded thresholds dict. Loaded from YAML if ``None``.

    Returns
    -------
    str
        One of ``"low"``, ``"nominal"``, or ``"high"``.
    """
    if thresholds is None:
        thresholds = _load_thresholds()
    ranges = thresholds["numerical_to_categorical"]

    if pd.isna(value):
        return "nominal"

    for label in ("low", "nominal", "high"):
        if ranges[label]["min"] <= value < ranges[label]["max"]:
            return label[0]  # l, n, or h
    return "h"


def categorical_to_numerical(value: str, thresholds: dict | None = None) -> float:
    """Convert a VIIRS categorical confidence (l/n/h) to a numerical value.

    Parameters
    ----------
    value:
        One of ``"l"``, ``"n"``, ``"h"`` (case-insensitive).
    thresholds:
        Optional pre-loaded thresholds dict.

    Returns
    -------
    float
        The representative numerical value.
    """
    if thresholds is None:
        thresholds = _load_thresholds()
    mapping = thresholds["categorical_to_numerical"]

    if pd.isna(value):
        return mapping["n"]

    value_lower = str(value).strip().lower()
    return mapping.get(value_lower, mapping["n"])


def add_unified_confidence(df: pd.DataFrame) -> pd.DataFrame:
    """Add unified ``confidence_cat`` and ``confidence_num`` columns.

    This function expects a merged FIRMS DataFrame with columns:
    - ``sensor``: ``"modis"``, ``"viirs_snpp"``, or ``"viirs_noaa20"``
    - ``confidence``: original confidence value (numerical for MODIS, categorical for VIIRS)

    It produces:
    - ``confidence_og_num``: original numerical value (from MODIS, ``NaN`` for VIIRS)
    - ``confidence_og_cat``: original categorical value (from VIIRS, ``NaN`` for MODIS)
    - ``confidence_cat``: unified categorical label
    - ``confidence_num``: unified numerical value

    Parameters
    ----------
    df:
        Merged FIRMS DataFrame.

    Returns
    -------
    pd.DataFrame
        DataFrame with the additional confidence columns.
    """
    df = df.copy()
    thresholds = _load_thresholds()

    df["confidence_og_num"] = pd.NA
    df["confidence_og_cat"] = pd.NA

    modis_mask = df["sensor"] == "modis"
    viirs_mask = df["sensor"].str.startswith("viirs")

    df.loc[modis_mask, "confidence_og_num"] = df.loc[modis_mask, "confidence"]
    df.loc[viirs_mask, "confidence_og_cat"] = df.loc[viirs_mask, "confidence"]

    df["confidence_num"] = pd.NA
    df["confidence_cat"] = pd.NA

    # MODIS: numerical → categorical
    df.loc[modis_mask, "confidence_num"] = pd.to_numeric(
        df.loc[modis_mask, "confidence"], errors="coerce"
    )
    df.loc[modis_mask, "confidence_cat"] = df.loc[modis_mask, "confidence_num"].apply(
        lambda v: numerical_to_categorical(v, thresholds)
    )

    # VIIRS: categorical → numerical
    df.loc[viirs_mask, "confidence_cat"] = df.loc[viirs_mask, "confidence"].str.lower()
    df.loc[viirs_mask, "confidence_num"] = df.loc[viirs_mask, "confidence_cat"].apply(
        lambda v: categorical_to_numerical(v, thresholds)
    )

    df["confidence_cat"] = df["confidence_cat"].astype("category")
    df["confidence_num"] = pd.to_numeric(df["confidence_num"], errors="coerce")

    return df


In [ ]:
df = add_unified_confidence(load_all_firms(country="Spain", years=[2023, 2024]))
display(df)
print(df.columns)

df_no_og = df.drop(columns=["confidence_og_num", "confidence_og_cat"])
display(df)

print(df_no_og.isnull().sum())

# test